# 🧠 Graph Metrics Extraction Pipeline (FA, GM, rsfMRI) – HCB & Naples Cohorts

This pipeline computes **global** and **nodal** graph metrics from subject-level connectivity matrices for the HCB & Naples cohort. It processes data from three brain connectivity layers:

- **FA - Fractional Anisotropy (structural)**
- **GM - Gray Matter Morphological Connectivity (morphological)**
- **rsfMRI - Resting-State Functional MRI (functional)**

Each subject’s connectivity matrix is stored as a `.csv` file and has been preprocessed (e.g., imputed, normalized, ComBat-harmonized). The filenames include the subject ID and the layer.

---

## 🔄 What the pipeline does

1. **Reads matrices** from folders, one `.csv` file per subject.
2. **Extracts subject ID** by removing the suffix from filenames (e.g., `_FA_corrected.csv`).
3. **Converts matrices to undirected weighted graphs** using NetworkX.
4. **Computes graph metrics**:
   - **Global metrics** (one per subject):
     - Graph density
     - Average clustering coefficient
     - Global efficiency
     - Average shortest path length (if connected)
   - **Nodal metrics** (per node, per subject):
     - Degree
     - Strength (weighted degree)
     - Betweenness centrality
     - Closeness centrality
     - Eigenvector centrality
5. **Saves results as Excel files** (`.xlsx`), one file per layer:
   - Global metrics: `global_graph_metrics_{layer}.xlsx`
   - Nodal metrics: `nodal_graph_metrics_{layer}.xlsx`

---


In [1]:
import os
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
#import matplotlib.pyplot as plt
#import seaborn as sns

# === GRAPH METRICS FUNCTIONS ===

def vector_to_matrix(vec, size=76):
    """
    Reconstructs a symmetric matrix (size x size) from the vectorized upper triangle.
    """
    mat = np.zeros((size, size))
    upper = np.triu_indices(size, k=1)
    mat[upper] = vec
    mat = mat + mat.T
    return mat

def matrix_to_graph(matrix):
    """
    Converts a NumPy connectivity matrix into an undirected weighted graph using NetworkX.
    """
    G = nx.from_numpy_array(matrix)
    return G

def compute_global_metrics(G):
    """
    Computes global-level graph metrics.
    Returns a dictionary with density, clustering, efficiency and shortest path.
    """
    return {
        "density": nx.density(G),
        "average_clustering": nx.average_clustering(G, weight='weight'),
        "global_efficiency": nx.global_efficiency(G),
        "average_shortest_path": nx.average_shortest_path_length(G) if nx.is_connected(G) else np.nan
    }

def compute_nodal_metrics(G):
    """
    Computes node-level graph metrics.
    Returns a DataFrame indexed by node with various nodal features.
    """
    nodal = pd.DataFrame(index=G.nodes)
    nodal['degree'] = pd.Series(dict(G.degree()))
    nodal['strength'] = pd.Series(dict(G.degree(weight='weight')))
    nodal['betweenness'] = pd.Series(nx.betweenness_centrality(G, weight='weight'))
    nodal['closeness'] = pd.Series(nx.closeness_centrality(G))
    nodal['eigenvector'] = pd.Series(nx.eigenvector_centrality_numpy(G, weight='weight'))
    return nodal

# === MAIN FUNCTION ===

def compute_graph_metrics_from_directory(input_dir: str, output_dir: str, layer_name: str, matrix_size: int = 76):
    """
    Computes graph metrics (global and nodal) from connectivity matrices stored in CSV files.
    Subject ID is extracted by removing the layer-specific suffix from the filename.

    Parameters:
    - input_dir: folder containing the CSV files for each subject
    - output_dir: destination folder to save the Excel results
    - layer_name: name of the layer (FA, GM, rsfMRI)
    - matrix_size: number of brain regions (default: 76)
    """
    os.makedirs(output_dir, exist_ok=True)

    global_metrics = []
    nodal_metrics = {}

    print(f"Processing layer: {layer_name}")

    for file in sorted(os.listdir(input_dir)):
        if file.endswith(".csv"):
            # Extract subject ID from filename by removing layer-specific suffix
            subj_id = file.replace("_FA_corrected.csv", "") \
                          .replace("_GM_corrected.csv", "") \
                          .replace("_rsfMRI_corrected.csv", "")

            matrix_path = os.path.join(input_dir, file)
            matrix = pd.read_csv(matrix_path, header=None).values

            if matrix.shape != (matrix_size, matrix_size):
                print(f"Skipping {file}: invalid shape {matrix.shape}")
                continue

            G = matrix_to_graph(matrix)

            # Global metrics
            gm = compute_global_metrics(G)
            gm['Subject'] = subj_id
            global_metrics.append(gm)

            # Nodal metrics
            nm = compute_nodal_metrics(G)
            nm['Subject'] = subj_id
            nodal_metrics[subj_id] = nm

    # Save results
    df_global = pd.DataFrame(global_metrics).set_index("Subject")
    df_nodal = pd.concat(nodal_metrics).reset_index().rename(columns={'level_0': 'Subject', 'level_1': 'Node'})

    df_global.to_excel(os.path.join(output_dir, f"global_graph_metrics_{layer_name}.xlsx"))
    df_nodal.to_excel(os.path.join(output_dir, f"nodal_graph_metrics_{layer_name}.xlsx"))

    print(f"Saved metrics for {layer_name} in {output_dir}")

# Funtion usage to compute graph metrics from directory
# Folder paths and parameters Naples
print("Processing Naples data...")
base_input = r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES"
base_output = os.path.join(base_input, "graph_metrics")  # Folder to save results

layers_info = {
    "FA": "corrected_FA_matrices_regression_re",
    "GM": "corrected_GM_matrices_regression_re",
    "rsfMRI": "corrected_rsfMRI_matrices_regression_re"
}

for layer_name, folder in layers_info.items():
    input_dir = os.path.join(base_input, folder)
    output_dir = os.path.join(base_output, folder.replace("corrected_", "graph-metrics_"))
    compute_graph_metrics_from_directory(input_dir, output_dir, layer_name=layer_name)

# Folder paths and parameters HCB
print("Processing HCB data...")
base_input = r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_HCB"
base_output = os.path.join(base_input, "graph_metrics")  # Folder to save results

layers_info = {
    "FA": "corrected_FA_matrices_regression_re",
    "GM": "corrected_GM_matrices_regression_re",
    "rsfMRI": "corrected_rsfMRI_matrices_regression_re"
}

for layer_name, folder in layers_info.items():
    input_dir = os.path.join(base_input, folder)
    output_dir = os.path.join(base_output, folder.replace("corrected_", "graph-metrics_"))
    compute_graph_metrics_from_directory(input_dir, output_dir, layer_name=layer_name)

Processing Naples data...
Processing layer: FA
Saved metrics for FA in F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES\graph_metrics\graph-metrics_FA_matrices_regression_re
Processing layer: GM
Saved metrics for GM in F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES\graph_metrics\graph-metrics_GM_matrices_regression_re
Processing layer: rsfMRI
Saved metrics for rsfMRI in F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES\graph_metrics\graph-metrics_rsfMRI_matrices_regression_re
Processing HCB data...
Processing layer: FA
Skipping fa_vectors.csv: invalid shape (164, 2851)
Saved metrics for FA in F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_HCB\graph_metrics\graph-metrics_FA_matrices_regression_re
Processing layer: GM
Saved metrics for GM in F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC

# 🧠 Multilayer Graph Metrics Pipeline (FA + GM + rsfMRI) – HCB & Naples Cohorts

This pipeline computes **global** and **nodal** graph metrics from subject-level **multilayer brain networks** that integrate:

- **FA (structural connectivity)** – used as inter-layer weights  
- **GM (morphological connectivity)** – intra-layer (layer 1)  
- **rsfMRI (functional connectivity)** – intra-layer (layer 2)  

All matrices are `.csv` files (`76x76`) and have been preprocessed (e.g., imputed, normalized to [0, 1], and ComBat-harmonized). Each file name includes the subject ID and the layer suffix (e.g., `_FA_corrected.csv`).

---

## 🔄 What this multilayer pipeline does

1. **Loads corrected matrices** from three folders per cohort (GM, rsfMRI, FA).
2. **Finds subjects** present in all three layers by matching filenames.
3. **Builds a multilayer supra-adjacency matrix** per subject:
   - Size: `152x152` (76 nodes × 2 layers)
   - Structure:
     ```
     [ GM     FA  ]
     [ FA   rsfMRI ]
     ```
4. **Converts the supra-matrix to a graph** (`networkx.Graph`)
5. **Computes graph theory metrics**:
   - 🔹 **Global metrics** (per subject):
     - Global efficiency
     - Average strength
     - Average clustering coefficient
     - Density
     - (Other metrics can be added)
   - 🔸 **Nodal metrics** (per region, per subject):
     - Strength (aggregated across layers)
     - Degree
     - Closeness centrality
     - Betweenness centrality
     - Local efficiency
6. **Saves results as Excel files**:
   - `global_graph_metrics_MULTILAYER.xlsx`
   - `nodal_graph_metrics_MULTILAYER.xlsx`

---

## ▶️ Example: Run for each cohort

```python
# Naples cohort
process_multilayer_graphs(
    base_dir=r"...\MULTILAYER\DADES_NAPLES",
    output_dir=r"...\MULTILAYER\DADES_NAPLES\graph_metrics\multilayer"
)

# HCB cohort
process_multilayer_graphs(
    base_dir=r"...\MULTILAYER\DADES_HCB",
    output_dir=r"...\MULTILAYER\DADES_HCB\graph_metrics\multilayer"
)


In [2]:
# === MULTILAYER PROCESSING FUNCTION === 

def build_supra_adjacency_matrix(gm, fmri, fa):
    """
    Builds a 152x152 supra-adjacency matrix from GM, rsfMRI and FA matrices.

    - gm: 76x76 morphological connectivity matrix (intra-layer 1)
    - fmri: 76x76 functional connectivity matrix (intra-layer 2)
    - fa: 76x76 structural connectivity matrix (used as inter-layer links)

    The resulting matrix is structured as:
    [ GM    FA ]
    [ FA    fMRI ]
    
    Returns:
        A 152x152 numpy array (supra-adjacency matrix)
    """
    N = gm.shape[0]
    M = np.zeros((2 * N, 2 * N)) # Initialize a 152x152 matrix

    # Intra-layer connections
    M[:N, :N] = gm           # GM layer
    M[N:, N:] = fmri         # rsfMRI layer

    # Inter-layer connections (FA)
    M[:N, N:] = fa           # FA connects GM -> rsfMRI
    M[N:, :N] = fa           # FA connects rsfMRI -> GM (symmetric)

    return M

def process_multilayer_graphs(base_dir, output_dir, matrix_size=76):
    """
    Computes multilayer graph metrics per subject using GM, rsfMRI and FA matrices.
    Only subjects with all three layers are processed.

    base_dir: path containing the 3 corrected matrix folders
    output_dir: path to save graph metrics
    """
    os.makedirs(output_dir, exist_ok=True)

    gm_dir = os.path.join(base_dir, "corrected_GM_matrices_regression_re")
    fmri_dir = os.path.join(base_dir, "corrected_rsfMRI_matrices_regression_re")
    fa_dir = os.path.join(base_dir, "corrected_FA_matrices_regression_re")

    def extract_ids(folder, suffix):
        return {
            f.replace(suffix, "") for f in os.listdir(folder)
            if f.endswith(".csv")
        }

    # Get intersection of subject IDs
    gm_ids = extract_ids(gm_dir, "_GM_corrected.csv")
    fmri_ids = extract_ids(fmri_dir, "_rsfMRI_corrected.csv")
    fa_ids = extract_ids(fa_dir, "_FA_corrected.csv")

    common_ids = sorted(gm_ids & fmri_ids & fa_ids)
    print(f"Found {len(common_ids)} subjects with all 3 layers.")

    global_metrics = []
    nodal_metrics = {}
    
    # Loop through each subject and process
    for subj_id in common_ids:
        gm = pd.read_csv(os.path.join(gm_dir, f"{subj_id}_GM_corrected.csv"), header=None).values
        fmri = pd.read_csv(os.path.join(fmri_dir, f"{subj_id}_rsfMRI_corrected.csv"), header=None).values
        fa = pd.read_csv(os.path.join(fa_dir, f"{subj_id}_FA_corrected.csv"), header=None).values

        supra_matrix = build_supra_adjacency_matrix(gm, fmri, fa)
        G = matrix_to_graph(supra_matrix)

        # Global
        gm_values = compute_global_metrics(G)
        gm_values["Subject"] = subj_id
        global_metrics.append(gm_values)

        # Nodal
        nm = compute_nodal_metrics(G)
        nm["Subject"] = subj_id
        nodal_metrics[subj_id] = nm

    # Save
    df_global = pd.DataFrame(global_metrics).set_index("Subject")
    df_nodal = pd.concat(nodal_metrics).reset_index().rename(columns={"level_0": "Subject", "level_1": "Node"})

    df_global.to_excel(os.path.join(output_dir, "global_graph_metrics_MULTILAYER.xlsx"))
    df_nodal.to_excel(os.path.join(output_dir, "nodal_graph_metrics_MULTILAYER.xlsx"))

    print("Multilayer graph metrics saved successfully.")

# Funtion usage to compute graph metrics from directory
# Folder paths and parameters Naples
print("Processing Naples data for multilayer graphs...")
process_multilayer_graphs(
    base_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES",
    output_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES\graph_metrics\multilayer"
)

# Folder paths and parameters HCB
print("Processing HCB data for multilayer graphs...")
process_multilayer_graphs(
    base_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_HCB",
    output_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_HCB\graph_metrics\multilayer"
)


Processing Naples data for multilayer graphs...
Found 105 subjects with all 3 layers.
Multilayer graph metrics saved successfully.
Processing HCB data for multilayer graphs...
Found 155 subjects with all 3 layers.
Multilayer graph metrics saved successfully.


# 🧠 MultiNetX Multilayer Graph Metrics Pipeline (FA + GM + rsfMRI)

This notebook cell defines a function `process_multilayer_multinetx()` that computes **global** and **nodal** graph‐theory metrics on subject‐level **multilayer brain networks** integrating:

- **FA (structural connectivity)** – used as inter‑layer weights  
- **GM (morphological connectivity)** – intra‑layer (layer 0)  
- **rsfMRI (functional connectivity)** – intra‑layer (layer 1)  

All input matrices are `76×76` CSVs, already imputed, normalized to [0, 1] and ComBat‑harmonized. Filenames include the subject ID plus layer suffix (e.g. `subj01_FA_corrected.csv`).

---

## 🔄 Pipeline Overview

1. **Load corrected matrices** from three directories per cohort:
   - `corrected_GM_matrices_median_re/`  
   - `corrected_rsfMRI_matrices_median_re/`  
   - `corrected_FA_matrices_median_re/`  

2. **Identify common subjects** by intersecting the set of IDs extracted from filenames ending in:
   - `_GM_corrected.csv`  
   - `_rsfMRI_corrected.csv`  
   - `_FA_corrected.csv`  

3. **Build a MultiNetX multilayer graph** per subject:
   - Create two intra‑layer `networkx.Graph` objects from the GM and rsfMRI matrices.
   - Construct a sparse inter‑layer adjacency block (`scipy.sparse.lil_matrix`) of size `152×152` with FA weights:
     ```python
     inter_block = lil_matrix((2*N, 2*N))
     inter_block[:N,    N:] = fa    # GM → rsfMRI
     inter_block[N:,    :N] = fa    # rsfMRI → GM
     ```
   - Instantiate with:
     ```python
     mg = MultilayerGraph(
         list_of_layers=[G_gm, G_fmri],
         inter_adjacency_matrix=inter_block
     )
     ```
   - Assign uniform weights:
     ```python
     mg.set_intra_edges_weights(layer=0, weight=intra_weight)
     mg.set_intra_edges_weights(layer=1, weight=intra_weight)
     mg.set_edges_weights(inter_layer_edges_weight=fa_weight)
     ```

4. **Flatten** the multilayer network:
   - The `mg` object itself behaves as a single `networkx.Graph` containing all intra‑ and inter‑layer edges.

5. **Compute graph metrics** on the flattened graph:
   - **Global (per subject)**  
     - Density  
     - Average clustering coefficient  
     - Global efficiency  
     - Average shortest path length (if connected)  
   - **Nodal (per node, per subject)**  
     - Degree  
     - Strength (weighted degree)  
     - Betweenness centrality  
     - Closeness centrality (using `distance='weight'`)  
     - Eigenvector centrality  

6. **Save outputs** as Excel files:
   - `global_graph_metrics_multinetx.xlsx`  
   - `nodal_graph_metrics_multinetx.xlsx`  

---

## ▶️ Example Usage

```python
# Naples cohort
process_multilayer_multinetx(
    base_dir=r"…\DADES_NAPLES",
    output_dir=r"…\DADES_NAPLES\graph_metrics\multilayer_multinetx",
    matrix_size=76,
    fa_weight=1.0,
    intra_weight=1.0
)

# HCB cohort
process_multilayer_multinetx(
    base_dir=r"…\DADES_HCB",
    output_dir=r"…\DADES_HCB\graph_metrics\multilayer_multinetx",
    matrix_size=76,
    fa_weight=1.0,
    intra_weight=1.0
)


In [3]:
# Import MultiNetX multilayer class directly
from multinetx.core.multilayer import MultilayerGraph
from scipy.sparse import lil_matrix

# === GRAPH METRICS FUNCTIONS ===

def compute_global_metrics(G):
    """
    Computes global-level graph metrics on a NetworkX graph.
    Returns a dictionary with density, clustering, efficiency, and average shortest path.
    """
    return {
        "density": nx.density(G),
        "average_clustering": nx.average_clustering(G, weight='weight'),
        "global_efficiency": nx.global_efficiency(G),
        "average_shortest_path": (
            nx.average_shortest_path_length(G) if nx.is_connected(G) else np.nan
        )
    }

def compute_nodal_metrics(G):
    """
    Computes node-level graph metrics for a NetworkX graph.
    Returns a DataFrame indexed by node with nodal features.
    """
    nodal = pd.DataFrame(index=G.nodes)
    nodal['degree'] = pd.Series(dict(G.degree()))
    nodal['strength'] = pd.Series(dict(G.degree(weight='weight')))
    nodal['betweenness'] = pd.Series(nx.betweenness_centrality(G, weight='weight'))
    # Use 'distance' parameter for weighted closeness
    nodal['closeness'] = pd.Series(nx.closeness_centrality(G, distance='weight'))
    nodal['eigenvector'] = pd.Series(nx.eigenvector_centrality_numpy(G, weight='weight'))
    return nodal

# === MULTILAYER PROCESSING FUNCTION ===

def process_multilayer_multinetx(base_dir: str, output_dir: str, matrix_size: int = 76, fa_weight: float = 1.0, intra_weight: float = 1.0):
    """
    Computes multilayer graph metrics per subject using MultiNetX.
    Only subjects with all three layers (GM, rsfMRI, FA) are processed.
    """
    base = Path(base_dir)
    gm_dir = base / "corrected_GM_matrices_regression_re"
    fmri_dir = base / "corrected_rsfMRI_matrices_regression_re"
    fa_dir = base / "corrected_FA_matrices_regression_re"
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    def extract_ids(folder: Path, suffix: str):
        return {f.stem.replace(suffix, '') for f in folder.glob(f"*{suffix}.csv")}

    common_ids = sorted(
        set(extract_ids(gm_dir, "_GM_corrected")) &
        set(extract_ids(fmri_dir, "_rsfMRI_corrected")) &
        set(extract_ids(fa_dir, "_FA_corrected"))
    )
    print(f"Found {len(common_ids)} subjects with all 3 layers.")

    global_metrics = []
    nodal_metrics = {}
    N = matrix_size

    for subj in common_ids:
        # Load matrices
        gm = pd.read_csv(gm_dir / f"{subj}_GM_corrected.csv", header=None).values
        fmri = pd.read_csv(fmri_dir / f"{subj}_rsfMRI_corrected.csv", header=None).values
        fa = pd.read_csv(fa_dir / f"{subj}_FA_corrected.csv", header=None).values

        # Build layer graphs
        G_gm = nx.from_numpy_array(gm)
        G_fmri = nx.from_numpy_array(fmri)

        # Build inter-layer adjacency as scipy lil_matrix
        inter_block = lil_matrix((2*N, 2*N))
        inter_block[:N, N:] = fa
        inter_block[N:, :N] = fa

        # Instantiate MultiNetX multilayer graph directly
        mg = MultilayerGraph(
            list_of_layers=[G_gm, G_fmri],
            inter_adjacency_matrix=inter_block
        )

        # Assign weights
        mg.set_intra_edges_weights(layer=0, weight=intra_weight)
        mg.set_intra_edges_weights(layer=1, weight=intra_weight)
        mg.set_edges_weights(inter_layer_edges_weight=fa_weight)

        # mg now contains all edges (intra + inter)
        G_flat = mg

        # Compute and store metrics
        gvals = compute_global_metrics(G_flat)
        gvals['Subject'] = subj
        global_metrics.append(gvals)
        ndf = compute_nodal_metrics(G_flat)
        ndf['Subject'] = subj
        nodal_metrics[subj] = ndf

    # Save to Excel
    df_global = pd.DataFrame(global_metrics).set_index('Subject')
    df_nodal = pd.concat(nodal_metrics).reset_index().rename(columns={'level_0': 'Subject', 'level_1': 'Node'})
    df_global.to_excel(out / "global_graph_metrics_multinetx.xlsx")
    df_nodal.to_excel(out / "nodal_graph_metrics_multinetx.xlsx")

    print(f"Saved multilayer metrics in {out}")

# Example usage:
# process_multilayer_multinetx(
#     base_dir=r"F:/.../DADES_NAPLES",
#     output_dir=r"F:/.../DADES_NAPLES/graph_metrics/multilayer_multinetx"
# )
print("Processing Naples data for multilayer graphs with MultiNetX...")
process_multilayer_multinetx(
     base_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES",
     output_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES\graph_metrics\multilayer_multinetx"
 )

print("Processing HCB data for multilayer graphs with MultiNetX...")
process_multilayer_multinetx(
     base_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_HCB",
     output_dir=r"F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_HCB\graph_metrics\multilayer_multinetx"
 )


Processing Naples data for multilayer graphs with MultiNetX...
Found 105 subjects with all 3 layers.
Saved multilayer metrics in F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_NAPLES\graph_metrics\multilayer_multinetx
Processing HCB data for multilayer graphs with MultiNetX...
Found 155 subjects with all 3 layers.
Saved multilayer metrics in F:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\DATOS\MULTILAYER\DADES_HCB\graph_metrics\multilayer_multinetx
